# RAG 系统 PDF 渲染 - 可执行的 Step-by-Step 指南

本指南从零开始，用纯 Python + mock 数据模拟完整的 RAG PDF 渲染流程：

```
Step 1: 生成 mock PDF 文件
Step 2: 生成 mock RAG chunks (带坐标)
Step 3: 模拟后端 API 返回二进制 PDF
Step 4: 前端加载 PDF 并提取页面尺寸
Step 5: 将 chunk 坐标转换为 PDF 高亮位置
Step 6: 渲染带高亮的 PDF (HTML 输出)
```

所有代码都可以直接在 notebook 中执行验证。

## 环境准备

In [1]:
!pip install reportlab 2>&1 | tail -3


In [2]:
import json, os
from io import BytesIO
from dataclasses import dataclass
from typing import List
print("Environment ready")


Environment ready


## Step 1: 生成 mock PDF 文件

用 reportlab 生成带文本的 PDF，模拟 RAG 知识库文档。

In [3]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import mm

def create_mock_pdf(output_path):
    c = canvas.Canvas(output_path, pagesize=A4)
    w, h = A4
    m = 50
    c.setFont("Helvetica-Bold", 24)
    c.drawString(m, h - 80, "RAG Knowledge Base Document")
    c.setFont("Helvetica", 12)
    y = h - 120
    lines = [
        ("Section 1: Introduction to RAG", True),
        ("Retrieval-Augmented Generation (RAG) combines information retrieval with", False),
        ("large language models to produce more accurate and grounded responses.", False),
        ("The key components include document parsing, text chunking, embedding", False),
        ("generation, vector storage, and retrieval.", False),
        ("", False),
        ("Section 2: Document Parsing", True),
        ("PDF documents are parsed using tools like PDFPlumber or PyMuPDF.", False),
        ("Each page is analyzed to extract text, tables, and bounding boxes.", False),
        ("The coordinates are stored as (x, y, width, height) in point units.", False),
        ("A4 page size is 595.27 x 841.89 points.", False),
        ("", False),
        ("Section 3: Chunking Strategy", True),
        ("Documents are split into overlapping chunks of 500-1000 tokens.", False),
        ("Each chunk retains its source metadata including page number and", False),
        ("bounding box coordinates for precise PDF highlighting.", False),
    ]
    for text, is_heading in lines:
        if y < m:
            c.showPage()
            y = h - 60
        c.setFont("Helvetica-Bold" if is_heading else "Helvetica", 14 if is_heading else 12)
        c.drawString(m, y, text)
        y -= 20
    c.showPage()
    c.setFont("Helvetica", 12)
    y = h - 60
    lines2 = [
        ("Section 4: Embedding and Retrieval", True),
        ("Text chunks are converted to dense vector embeddings using models like", False),
        ("SentenceTransformers or OpenAI embeddings. These vectors are stored in", False),
        ("a vector database (e.g., Milvus, Pinecone, FAISS) for similarity search.", False),
        ("", False),
        ("Section 5: PDF Highlighting", True),
        ("When a retrieved chunk is displayed, its bounding box coordinates are", False),
        ("converted to relative positions (x/page_width, y/page_height) and used", False),
        ("to render highlight overlays on the PDF viewer.", False),
        ("", False),
        ("Section 6: Frontend Rendering", True),
        ("The frontend uses PDF.js to render PDF pages. Highlight overlays are", False),
        ("positioning using the relative coordinates calculated from the chunk", False),
        ("bounding boxes and the PDF page dimensions.", False),
        ("", False),
        ("End of Document", True),
    ]
    for text, is_heading in lines2:
        if y < m:
            c.showPage()
            y = h - 60
        c.setFont("Helvetica-Bold" if is_heading else "Helvetica", 14 if is_heading else 12)
        c.drawString(m, y, text)
        y -= 20
    c.save()
    return output_path

os.makedirs("mock_output", exist_ok=True)
pdf_path = create_mock_pdf("mock_output/knowledge_base.pdf")
print(f"PDF created: {pdf_path}, size: {os.path.getsize(pdf_path)} bytes")


PDF created: mock_output/knowledge_base.pdf, size: 3205 bytes


In [4]:
pdf_size = os.path.getsize(pdf_path)
print(f"PDF size: {pdf_size} bytes ({pdf_size/1024:.1f} KB)")
assert pdf_size > 0
print("PDF verification PASSED")


PDF size: 3205 bytes (3.1 KB)
PDF verification PASSED


## Step 2: 生成 mock RAG Chunks

模拟 RAG 系统从 PDF 中提取的 chunks，每个 chunk 包含文本、页码和 bounding box 坐标。

In [5]:
@dataclass
class BoundingBox:
    x: float
    y: float
    width: float
    height: float
    def to_dict(self):
        return {"x": self.x, "y": self.y, "w": self.width, "h": self.height}

@dataclass
class ChunkPosition:
    page_number: int
    bbox: BoundingBox

@dataclass
class RAGChunk:
    chunk_id: str
    content: str
    positions: List[ChunkPosition]
    doc_id: str = "mock-doc-001"
    def to_dict(self):
        return {
            "chunk_id": self.chunk_id,
            "content": self.content[:80] + "..." if len(self.content) > 80 else self.content,
            "doc_id": self.doc_id,
            "positions": [{"page": p.page_number, "bbox": p.bbox.to_dict()} for p in self.positions]
        }

A4_W, A4_H = 595.27, 841.89
m = 50
mock_chunks = [
    RAGChunk("chunk-001", "Retrieval-Augmented Generation (RAG) combines information retrieval with large language models to produce more accurate and grounded responses.", [ChunkPosition(1, BoundingBox(m, A4_H - 120, A4_W - 2*m, 80))]),
    RAGChunk("chunk-002", "PDF documents are parsed using tools like PDFPlumber or PyMuPDF. Each page is analyzed to extract text, tables, and bounding boxes.", [ChunkPosition(1, BoundingBox(m, A4_H - 300, A4_W - 2*m, 80))]),
    RAGChunk("chunk-003", "Text chunks are converted to dense vector embeddings using models like SentenceTransformers or OpenAI embeddings.", [ChunkPosition(2, BoundingBox(m, A4_H - 60, A4_W - 2*m, 80))]),
    RAGChunk("chunk-004", "The frontend uses PDF.js to render PDF pages. Highlight overlays are positioned using relative coordinates.", [ChunkPosition(2, BoundingBox(m, A4_H - 220, A4_W - 2*m, 80))]),
]
print("=== Mock RAG Chunks ===")
for i, c in enumerate(mock_chunks):
    d = c.to_dict()
    print(f"Chunk {i+1}: {d['chunk_id']} | Page {d['positions'][0]['page']} | BBox: {d['positions'][0]['bbox']}")


=== Mock RAG Chunks ===
Chunk 1: chunk-001 | Page 1 | BBox: {'x': 50, 'y': 721.89, 'w': 495.27, 'h': 80}
Chunk 2: chunk-002 | Page 1 | BBox: {'x': 50, 'y': 541.89, 'w': 495.27, 'h': 80}
Chunk 3: chunk-003 | Page 2 | BBox: {'x': 50, 'y': 781.89, 'w': 495.27, 'h': 80}
Chunk 4: chunk-004 | Page 2 | BBox: {'x': 50, 'y': 621.89, 'w': 495.27, 'h': 80}


## Step 3: 模拟后端 API

模拟 `/api/v1/documents/{doc_id}/preview` 端点，返回二进制 PDF 数据。

In [6]:
class MockBackendAPI:
    def __init__(self, pdf_path, chunks):
        self.chunks = {c.chunk_id: c for c in chunks}
        self.doc_id = "mock-doc-001"
        with open(pdf_path, "rb") as f:
            self.pdf_binary = f.read()
    def get_pdf_preview(self, doc_id, auth_token="mock-token"):
        if doc_id != self.doc_id:
            return {"status": 404}
        return {"status": 200, "headers": {"Content-Type": "application/pdf", "Content-Length": len(self.pdf_binary)}, "data": self.pdf_binary}
    def get_chunk_highlights(self, chunk_id):
        if chunk_id not in self.chunks:
            return {"status": 404}
        c = self.chunks[chunk_id]
        return {"status": 200, "data": {"chunk_id": c.chunk_id, "content": c.content, "positions": c.to_dict()["positions"]}}

api = MockBackendAPI(pdf_path, mock_chunks)
resp = api.get_pdf_preview("mock-doc-001")
print(f"PDF API: {resp['status']} | Data: {len(resp['data'])} bytes | Magic: {resp['data'][:5]}")
assert resp["status"] == 200 and resp["data"].startswith(b"%PDF-")
print("API verification PASSED")


PDF API: 200 | Data: 3205 bytes | Magic: b'%PDF-'
API verification PASSED


In [7]:
resp = api.get_chunk_highlights("chunk-001")
print(f"Chunk API: {resp['status']}")
print(json.dumps(resp["data"], indent=2))
assert resp["status"] == 200
print("Chunk API PASSED")


Chunk API: 200
{
  "chunk_id": "chunk-001",
  "content": "Retrieval-Augmented Generation (RAG) combines information retrieval with large language models to produce more accurate and grounded responses.",
  "positions": [
    {
      "page": 1,
      "bbox": {
        "x": 50,
        "y": 721.89,
        "w": 495.27,
        "h": 80
      }
    }
  ]
}
Chunk API PASSED


## Step 4: 前端加载 PDF 并提取页面尺寸

模拟 PDF.js 加载 PDF 后获取页面尺寸。

In [8]:
@dataclass
class PdfDocument:
    num_pages: int
    page_width: float
    page_height: float
    title: str
    def get_page_info(self, n):
        return {"pageNumber": n, "width": self.page_width, "height": self.page_height}

class MockPdfLoader:
    @staticmethod
    def load_pdf(data):
        return PdfDocument(2, A4_W, A4_H, "RAG Knowledge Base Document")

pdf_doc = MockPdfLoader.load_pdf(api.get_pdf_preview("mock-doc-001")["data"])
print(f"PDF: {pdf_doc.title} | Pages: {pdf_doc.num_pages}")
for pg in range(1, pdf_doc.num_pages + 1):
    info = pdf_doc.get_page_info(pg)
    print(f"  Page {pg}: {info['width']:.1f} x {info['height']:.1f} pts ({info['width']/mm:.0f} x {info['height']/mm:.0f} mm)")
print("PDF load PASSED")


PDF: RAG Knowledge Base Document | Pages: 2
  Page 1: 595.3 x 841.9 pts (210 x 297 mm)
  Page 2: 595.3 x 841.9 pts (210 x 297 mm)
PDF load PASSED


## Step 5: 坐标转换 - chunk bbox -> 归一化高亮位置

核心步骤：绝对坐标 -> 相对坐标 (0~1)

In [9]:
@dataclass
class HighlightPosition:
    page_number: int
    x: float
    y: float
    width: float
    height: float
    def to_dict(self):
        return {"pageNumber": self.page_number, "x": round(self.x, 4), "y": round(self.y, 4), "width": round(self.width, 4), "height": round(self.height, 4)}

@dataclass
class Highlight:
    position: HighlightPosition
    content: dict
    chunk_id: str
    def to_dict(self):
        return {"chunk_id": self.chunk_id, "position": self.position.to_dict(), "content": self.content}

def build_chunk_highlights(chunk, pw, ph):
    result = []
    for pos in chunk.positions:
        hp = HighlightPosition(
            page_number=pos.page_number,
            x=pos.bbox.x / pw,
            y=pos.bbox.y / ph,
            width=pos.bbox.width / pw,
            height=pos.bbox.height / ph,
        )
        result.append(Highlight(hp, {"emoji": "\U0001F4CC", "text": "Chunk: " + chunk.chunk_id}, chunk.chunk_id))
    return result

print("=== Coordinate Conversion ===")
all_hl = []
for chunk in mock_chunks:
    hls = build_chunk_highlights(chunk, pdf_doc.page_width, pdf_doc.page_height)
    all_hl.extend(hls)
    pos = chunk.positions[0]
    p = hls[0].position
    print(f"{chunk.chunk_id}: raw({pos.bbox.x:.0f},{pos.bbox.y:.0f},{pos.bbox.width:.0f},{pos.bbox.height:.0f}) -> norm({p.to_dict()})")
print(f"Total highlights: {len(all_hl)}")
print("Conversion PASSED")


=== Coordinate Conversion ===
chunk-001: raw(50,722,495,80) -> norm({'pageNumber': 1, 'x': 0.084, 'y': 0.8575, 'width': 0.832, 'height': 0.095})
chunk-002: raw(50,542,495,80) -> norm({'pageNumber': 1, 'x': 0.084, 'y': 0.6437, 'width': 0.832, 'height': 0.095})
chunk-003: raw(50,782,495,80) -> norm({'pageNumber': 2, 'x': 0.084, 'y': 0.9287, 'width': 0.832, 'height': 0.095})
chunk-004: raw(50,622,495,80) -> norm({'pageNumber': 2, 'x': 0.084, 'y': 0.7387, 'width': 0.832, 'height': 0.095})
Total highlights: 4
Conversion PASSED


## Step 6: HTML 可视化 - 渲染带高亮的 PDF

生成 HTML 文件，CSS 模拟 PDF 页面+高亮。

In [10]:
def generate_html(highlights, pw, ph, out="mock_output/highlight_visualization.html"):
    by_page = {}
    for h in highlights:
        by_page.setdefault(h.position.page_number, []).append(h)

    pages = ""
    for pg in sorted(by_page):
        hls_html = ""
        for h in by_page[pg]:
            p = h.position
            style = "position:absolute;left:{}%;top:{}%;width:{}%;height:{}%;background:rgba(255,255,0,0.35);border:2px solid #FFD700;border-radius:4px;".format(p.x*100, p.y*100, p.width*100, p.height*100)
            hls_html += '<div class="highlight" style="{}" title="{}: {}"><span class="label">{}</span></div>'.format(style, h.chunk_id, h.content["text"], h.chunk_id)

        if pg == 1:
            content = '<h2>RAG Knowledge Base Document</h2><b>Section 1: Intro</b><br>RAG combines retrieval with LLMs.<br><b>Section 2: Parsing</b><br>PDF parsed with bbox coords.<br><b>Section 3: Chunking</b><br>Chunks keep page+bbox metadata.'
        else:
            content = '<b>Section 4: Embedding</b><br>Vectors stored in FAISS/Milvus.<br><b>Section 5: Highlighting</b><br>Bbox -> relative coords.<br><b>Section 6: Rendering</b><br>PDF.js + highlight overlays.'

        ps = 'position:relative;width:{}px;height:{}px;background:white;border:1px solid #ccc;margin:20px auto;box-shadow:0 2px 8px rgba(0,0,0,0.15);'.format(pw, ph)
        pages += '<div style="{}"><div style="position:absolute;top:10px;left:20px;font-weight:bold;">Page {}</div><div style="padding:60px 50px;font-family:Arial;font-size:14px;line-height:1.6;">{}</div>{}</div>'.format(ps, pg, content, hls_html)

    legend = "".join('<li><b>{}</b>: Page {}</li>'.format(h.chunk_id, h.position.page_number) for h in highlights)

    html = '<!DOCTYPE html><html><head><meta charset="utf-8"><title>RAG PDF Highlights</title>'
    html += '<style>body{background:#f0f0f0;font-family:Arial;padding:20px}.container{max-width:800px;margin:0 auto}h1{text-align:center}.legend{background:white;padding:15px;border-radius:8px;margin:20px 0}.highlight:hover{background:rgba(255,200,0,0.6)!important;cursor:pointer}.label{position:absolute;bottom:-18px;left:0;font-size:11px;color:#b8860b}</style></head>'
    html += '<body><div class="container"><h1>RAG PDF Highlight Visualization</h1>'
    html += '<div class="legend"><h3>Legend</h3><p>Yellow rectangles show RAG-retrieved chunk positions in the PDF.</p><ul>' + legend + '</ul></div>'
    html += pages
    html += '</div></body></html>'

    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    with open(out, "w") as f:
        f.write(html)
    return out

html_path = generate_html(all_hl, pdf_doc.page_width, pdf_doc.page_height)
print(f"HTML generated: {html_path}, size: {os.path.getsize(html_path)}")


HTML generated: mock_output/highlight_visualization.html, size: 3044


In [11]:
from IPython.display import IFrame
try:
    display(IFrame(html_path, width=800, height=600))
except:
    print("Open:", os.path.abspath(html_path))


## Step 7: 端到端验证

完整 RAG 查询->检索->高亮流程。

In [12]:
class MockRAGPipeline:
    def __init__(self, api, pdf_doc):
        self.api = api
        self.pdf_doc = pdf_doc
    def query(self, question):
        q = question.lower()
        scored = []
        for c in self.api.chunks.values():
            s = sum(1 for w in q.split() if w in c.content.lower())
            scored.append((s, c))
        scored.sort(key=lambda x: -x[0])
        top = [c for _, c in scored[:2]]
        print(f"Query: {question} -> found {len(top)} chunks")
        for c in top:
            print(f"  {c.chunk_id}: {c.content[:60]}...")
        hl = []
        for c in top:
            hl.extend(build_chunk_highlights(c, self.pdf_doc.page_width, self.pdf_doc.page_height))
        return {"question": question, "pdf_url": "/api/v1/documents/" + self.api.doc_id + "/preview",
                "chunks": [c.to_dict() for c in top], "highlights": [h.to_dict() for h in hl]}

pipe = MockRAGPipeline(api, pdf_doc)
r1 = pipe.query("How does PDF highlighting work?")
print(f"Highlights: {len(r1['highlights'])}")
for h in r1["highlights"]:
    p = h["position"]
    print(f"  Page {p['pageNumber']}: x={p['x']} y={p['y']} w={p['width']} h={p['height']}")
print()
r2 = pipe.query("What is RAG and embedding?")
print(f"Highlights: {len(r2['highlights'])}")
print("\nEND-TO-END PASSED")


Query: How does PDF highlighting work? -> found 2 chunks
  chunk-002: PDF documents are parsed using tools like PDFPlumber or PyMu...
  chunk-004: The frontend uses PDF.js to render PDF pages. Highlight over...
Highlights: 2
  Page 1: x=0.084 y=0.6437 w=0.832 h=0.095
  Page 2: x=0.084 y=0.7387 w=0.832 h=0.095

Query: What is RAG and embedding? -> found 2 chunks
  chunk-001: Retrieval-Augmented Generation (RAG) combines information re...
  chunk-002: PDF documents are parsed using tools like PDFPlumber or PyMu...
Highlights: 2

END-TO-END PASSED


## 总结

### 流程
```
Step 1: mock PDF (reportlab)
Step 2: mock RAG chunks (bbox coords)
Step 3: mock API -> binary PDF
Step 4: PDF load -> page dimensions
Step 5: bbox -> normalized coords
Step 6: HTML visualization
Step 7: end-to-end query
```

### 坐标转换
```python
x_rel = bbox.x / page_width
y_rel = bbox.y / page_height
```

### Mock -> Real
| Mock | Real |
|------|------|
| MockBackendAPI | FastAPI /api/v1/documents/{id}/preview |
| MockPdfLoader | PDF.js PdfLoader |
| build_chunk_highlights | useGetChunkHighlights |
| MockRAGPipeline | RAG retrieval pipeline |
| HTML viz | React PdfPreview + PdfHighlighter |
